# 07 · Reading a plate by redrawing it

The YOLO26s reader is kept as it is. After it, the 5 best valid readings of a plate are drawn as clean plates,
fitted to the photo (corners, brightness, blur), and the reading whose drawing fits best is chosen.

**Inputs:** the `bangla-plate-reading` dataset and the YOLO26s reader `best.pt`.

In [ ]:
%pip install -q ultralytics==8.4.46

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

DATA = next(p for p in INPUT.rglob("bangla-plate-reading") if p.is_dir())
WEIGHTS = list(INPUT.rglob("best.pt"))
print(DATA)
print(WEIGHTS)

In [ ]:
classes = json.loads((DATA / "classes.json").read_text(encoding="utf-8"))
bangla = [c["bangla"] for c in classes]
kind = [c["kind"] for c in classes]

test_images = sorted((DATA / "images" / "test").iterdir())
print(len(classes), "classes")
print(len(test_images), "test plates")

## Plate text

Same as notebooks 05 and 06: district, word and letter from left to right, then the six digits as `xx-xxxx`.
The true text of a test plate comes from its label file.

In [ ]:
def plate_text(chars):
    # chars: list of (class id, x centre)
    chars = sorted(chars, key=lambda c: c[1])
    head = " ".join(bangla[c] for c, x in chars if kind[c] != "digit")
    digits = "".join(bangla[c] for c, x in chars if kind[c] == "digit")
    if len(digits) == 6:
        digits = digits[:2] + "-" + digits[2:]
    return " ".join(part for part in [head, digits] if part)


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]


def read_labels(image_path, split="test"):
    # YOLO label file -> list of (class id, x centre, y centre, width, height)
    label = DATA / "labels" / split / (image_path.stem + ".txt")
    boxes = []
    for line in label.read_text().splitlines():
        p = line.split()
        if p:
            boxes.append((int(p[0]), *map(float, p[1:])))
    return boxes


def read_truth(image_path, split="test"):
    return plate_text([(b[0], b[1]) for b in read_labels(image_path, split)])


for path in test_images[:3]:
    print(path.name[:50], "->", read_truth(path))
print(edit_distance("ঢাকা মেট্রো গ ২৪-২৯৫৫", "ঢাকা মেট্রো ২৪-২৯৫৫"))

## Baseline: current reader on the test set

The reader at 0.25 confidence, characters sorted left to right, as in the deployed AI service.
This must match notebook 05: 77.4% on all 939 plates (75.7% bd-lpr, 77.9% blpd).

In [ ]:
reader = YOLO(WEIGHTS[0])
device = 0 if torch.cuda.is_available() else "cpu"

rows = []
for start in range(0, len(test_images), 64):
    batch = test_images[start:start + 64]
    results = reader.predict(batch, imgsz=320, conf=0.25, device=device, verbose=False)
    for path, r in zip(batch, results):
        chars = list(zip(r.boxes.cls.int().tolist(), r.boxes.xywhn[:, 0].tolist()))
        truth = read_truth(path)
        pred = plate_text(chars)
        rows.append({"image": path.name, "source": path.name.split("__")[0],
                     "truth": truth, "predicted": pred, "correct": truth == pred,
                     "errors": edit_distance(truth, pred), "length": len(truth)})

baseline = pd.DataFrame(rows)
baseline.head()

In [ ]:
def summary(df):
    return pd.Series({"plates": len(df),
                      "plate acc %": round(100 * df["correct"].mean(), 1),
                      "CER %": round(100 * df["errors"].sum() / df["length"].sum(), 2)})


table = baseline.groupby("source").apply(summary)
table.loc["all"] = summary(baseline)
table